#### 1. Load Data and Define Parameters

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# Load the data generated
df = pd.read_csv('../data/raw/simulated_inventory_base.csv')

# Constants for Supply Chain Logic
SERVICE_LEVEL = 0.95 
Z_SCORE = stats.norm.ppf(SERVICE_LEVEL)

print(f"Data Loaded. Service Level set to {SERVICE_LEVEL*100}% (Z-Score: {Z_SCORE:.3f})")

Data Loaded. Service Level set to 95.0% (Z-Score: 1.645)


#### 2. ABC Classification (The Pareto Principle)

In [2]:
# 1. Sort by Annual Consumption Value (Daily Consumption * 365)
df['Annual_Consumption_USD'] = df['Daily_Consumption_USD'] * 365
df = df.sort_values(by='Annual_Consumption_USD', ascending=False)

# 2. Calculate Cumulative Percentage
df['Cum_Consumption_USD'] = df['Annual_Consumption_USD'].cumsum()
df['Cum_Percentage'] = 100 * df['Cum_Consumption_USD'] / df['Annual_Consumption_USD'].sum()

# 3. Assign Classes
def abc_classify(pct):
    if pct <= 70: return 'A (High Value)'
    elif pct <= 90: return 'B (Medium Value)'
    else: return 'C (Low Value)'

df['ABC_Class'] = df['Cum_Percentage'].apply(abc_classify)

print("ABC Classification Complete. Distribution:")
print(df['ABC_Class'].value_counts())

ABC Classification Complete. Distribution:
ABC_Class
A (High Value)      6
C (Low Value)       5
B (Medium Value)    4
Name: count, dtype: int64


#### 3. Calculating Safety Stock (The Buffer)

In [3]:
# We take the square root of Lead Time because uncertainty grows over time
df['Safety_Stock'] = (Z_SCORE * df['Demand_Volatility'] * np.sqrt(df['Lead_Time_Days'])).round(0)

# Ensure no negative safety stock
df['Safety_Stock'] = df['Safety_Stock'].clip(lower=0)

print("Safety Stock calculated based on demand volatility and import lead times.")

Safety Stock calculated based on demand volatility and import lead times.


#### 4. Reorder Point (The Trigger)

In [4]:
# Calculate Reorder Point
df['Reorder_Point'] = ((df['Avg_Daily_Demand'] * df['Lead_Time_Days']) + df['Safety_Stock']).round(0)

# Identify Stock Status
def check_status(row):
    if row['Initial_Stock'] <= row['Reorder_Point']:
        return 'REORDER NOW'
    elif row['Initial_Stock'] <= (row['Reorder_Point'] * 1.2):
        return 'WATCHLIST'
    else:
        return 'HEALTHY'

df['Stock_Status'] = df.apply(check_status, axis=1)

print("Reorder Points established. Ready for decision support.")
df[['Product_Name', 'ABC_Class', 'Initial_Stock', 'Reorder_Point', 'Stock_Status']].head(10)

Reorder Points established. Ready for decision support.


,Product_Name,ABC_Class,Initial_Stock,Reorder_Point,Stock_Status
11,UV Sterilizer,A (High Value),16,140.0,REORDER NOW
5,5kVA Hybrid Inverter,A (High Value),54,80.0,REORDER NOW
1,Submersible Pump,A (High Value),44,96.0,REORDER NOW
14,FRP Vessel,A (High Value),20,99.0,REORDER NOW
12,RO Membrane,A (High Value),33,13.0,HEALTHY
3,Booster Pump,A (High Value),45,120.0,REORDER NOW
10,50kVA Silent Gen,B (Medium Value),12,118.0,REORDER NOW
9,3kVA Petrol Gen,B (Medium Value),50,100.0,REORDER NOW
7,Solar Controller,B (Medium Value),32,96.0,REORDER NOW
0,Centrifugal Pump,B (Medium Value),24,87.0,REORDER NOW


#### 5. Save Processed Data

In [5]:
processed_path = '../data/processed/inventory_intelligence.csv'
df.to_csv(processed_path, index=False)

print(f"Processed data saved to: {processed_path}")

Processed data saved to: ../data/processed/inventory_intelligence.csv
